# KNN (K-Nearest-Neighbors)

KNN is a simple concept: define some distance metric between the items in your dataset, and find the K closest items. You can then use those items to predict some property of a test item, by having them somehow “vote” on it.

As an example, let’s look at the MovieLens data. We’ll try to guess the rating of a movie by looking at the 10 movies that are closest to it in terms of genres and popularity.

To start, we’ll load up every rating in the data set into a Pandas DataFrame:

In [48]:
import numpy as np
import pandas as pd
#aqui listamos os filmes
r_cols = ['user_id', 'movie_id', 'rating']
ratings = pd.read_csv('C:/Users/ganog/Downloads/Aquivos_de_Codigo/u.data',
                      sep='\t', names=r_cols, usecols=range(3))
ratings.head()

,user_id,movie_id,rating
0,0,50,5
1,0,172,5
2,0,133,1
3,196,242,3
4,186,302,3


In [49]:
movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]})
movieProperties.head() #agregamos pela nota gera do filme

rating          
           size      mean
movie_id                 
1           452  3.878319
2           131  3.206107
3            90  3.033333
4           209  3.550239
5            86  3.302326

In [ ]:
#oiii fernando

In [50]:
movieNumRatings = pd.DataFrame(movieProperties['rating']['size'])
movieNormalizeNumRatings = movieNumRatings.apply(lambda x: (x - np.min(x))/(np.max(x) - np.min(x)))
movieNormalizeNumRatings.head() #os mais populares se aproximam de 1 - função lambda explica melhor essa conta.

,size
movie_id,
1,0.773585
2,0.222985
3,0.152659
4,0.356775
5,0.145798


In [55]:
movieDict = { }
with open (r'C:/Users/ganog/Downloads/Aquivos_de_Codigo/u.item') as f:
    temp = ''
    for line in f:
        fields = line.rstrip('\n').split('|')
        movieID = int(fields[0])
        name = fields[1]
        genres = fields[5:25]
        genres = list(map(int, genres)) #aqui coloquei o list para aparecer bonitinho e não dar erro no código ComputeDistance, mas no geral, esse código cria um catálogo que tem informações sobre cada filme separado em nome, genero, avaliação e quantas pessoas já viram
        movieDict[movieID] = (name, genres, movieNormalizeNumRatings.loc[movieID].get('size'), movieProperties.loc[movieID].rating)

In [56]:
movieDict[1]

('Toy Story (1995)',
 [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 np.float64(0.7735849056603774),
 size    452.000000
 mean      3.878319
 Name: 1, dtype: float64)

In [57]:
from scipy import spatial

def ComputeDistance(a, b):
    genresA = a[1]
    genresB = b[1]
    genreDistance = spatial.distance.cosine(genresA, genresB)
    popularityA = a[2]
    popularityB = b[2]
    popularityDistance = abs(popularityA - popularityB)
    return genreDistance + popularityDistance

ComputeDistance(movieDict[2], movieDict[4]) # quão próximo os filmes são, aqui compara os generos e a popularidade

np.float64(0.8004574042309892)

In [59]:
print(movieDict[2])
print(movieDict[4])

('GoldenEye (1995)', [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], np.float64(0.22298456260720412), size    131.000000
mean      3.206107
Name: 2, dtype: float64)
('Get Shorty (1995)', [0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], np.float64(0.3567753001715266), size    209.000000
mean      3.550239
Name: 4, dtype: float64)


In [64]:
import operator
#Essa função faz a mesma da passada, mas pega os filmes "vizinhos", os 10 mais próximos nesse caso. Compara a média entre os dois e verifica se é vizinho.
def getNeighbors(movieID, K):
    distances = []

    for movie in movieDict:
        if (movie != movieID):
            dist = ComputeDistance(movieDict[movieID], movieDict[movie])
            distances.append((movie, dist))

    distances.sort(key=operator.itemgetter(1))

    neighbors = []

    for x in range(K):
        neighbors.append(distances[x][0])

    return neighbors

K = 10
avgRating = 0
neighbors = getNeighbors(1, K)

for neighbor in neighbors:
    avgRating += movieDict[neighbor][3]
    print(movieDict[neighbor][0] + " " + str(movieDict[neighbor][3]))

avgRating /= float(K)

#os resultados aparecem um pouco diferente do professor, isso se deve ao fato da versão do python e do Pycharm

Liar Liar (1997) size    485.000000
mean      3.156701
Name: 294, dtype: float64
Aladdin (1992) size    219.000000
mean      3.812785
Name: 95, dtype: float64
Willy Wonka and the Chocolate Factory (1971) size    326.000000
mean      3.631902
Name: 151, dtype: float64
Monty Python and the Holy Grail (1974) size    316.000000
mean      4.066456
Name: 168, dtype: float64
Full Monty, The (1997) size    315.000000
mean      3.926984
Name: 269, dtype: float64
George of the Jungle (1997) size    162.000000
mean      2.685185
Name: 259, dtype: float64
Beavis and Butt-head Do America (1996) size    156.000000
mean      2.788462
Name: 240, dtype: float64
Birdcage, The (1996) size    293.000000
mean      3.443686
Name: 25, dtype: float64
Home Alone (1990) size    137.000000
mean      3.087591
Name: 94, dtype: float64
Aladdin and the King of Thieves (1996) size    26.000000
mean     2.846154
Name: 422, dtype: float64


In [65]:
avgRating

size    243.500000
mean      3.344591
Name: 294, dtype: float64

In [66]:
movieDict[1] #nas buscas vão aparecer size e mean pelo mesmo motivo anterior, versão. Mas os valores principais aparecem, aqui só voltamos ao valor do Toy Story.

('Toy Story (1995)',
 [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 np.float64(0.7735849056603774),
 size    452.000000
 mean      3.878319
 Name: 1, dtype: float64)